[요구사항] 고객 구매 여부 예측

- 학습용 데이터를 이용하여 purchase를 목표변수로 설정하고, 목표변수에 레이블 인코딩을 적용하시오. 독립변수는 별도의 인코딩 없이 사용하고, 랜덤포레스트 분류 모델을 학습하시오.
- train_test_split()으로 학습·검증 데이터를 분할하고, 랜덤포레스트 분류 모델의 평가지표 5가지를 출력
- 학습한 모델을 평가용 데이터에 적용하여 구매 여부를 예측하고, 예측 결과를 pred 컬럼으로 갖는 데이터프레임을 생성한 후 result.csv로 저장하시오.

​

[purchase_train.csv 데이터셋 구조]
| 컬럼 | 의미 | 처리 |
|---|---|---|
| `age` | 나이 | 숫자형 그대로 |
| `income` | 월 소득(만원) | 숫자형 그대로 |
| `membership` | 회원 등급 | 인코딩 없이 사용 |
| `purchase` | 구매 여부 | 목표변수 `y`, 레이블 인코딩 |
| `pred` | 예측 결과 | 제출용 컬럼 |

- 분류 알고리즘: RandomForestClassifier
- 목표변수: purchase (아니오, 예)
- 평가용 데이터의 정답은 제공하지 않음.
- 결과 파일: result.csv

<br>

- 독립변수 인코딩 없음: 숫자형 컬럼만 사용하므로 별도 인코딩 불필요.
- 종속변수 인코딩: LabelEncoder()로 문자열 정답을 숫자로 변환.
- 분류 모델: RandomForestClassifier().
- 예측: model.predict(X_test).
- 레이블 복원: le.inverse_transform(pred).
- CSV 저장: to_csv("result.csv", index=False).

In [1]:
import pandas as pd

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

# 1. CSV 파일 불러오기
train = pd.read_csv("purchase_train.csv")
test = pd.read_csv("purchase_test.csv")

In [2]:
# 2. 데이터 기본 정보 확인
display('데이터 기본 정보 확인')
display(train.head())
display(train.tail())
display(train.shape)
display(train.info())

'데이터 기본 정보 확인'

,age,income,visits,purchase
0,22,220,2,아니오
1,25,250,3,아니오
2,28,280,4,아니오
3,35,350,5,예
4,40,400,6,예


,age,income,visits,purchase
15,31,310,4,아니오
16,37,370,6,예
17,44,440,7,예
18,26,260,3,아니오
19,52,520,9,예


(20, 4)

<class 'pandas.DataFrame'>
RangeIndex: 20 entries, 0 to 19
Data columns (total 4 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   age       20 non-null     int64
 1   income    20 non-null     int64
 2   visits    20 non-null     int64
 3   purchase  20 non-null     str  
dtypes: int64(3), str(1)
memory usage: 880.0 bytes


None

In [3]:
# 3. 기술통계 확인
display('기술통계 확인')
display(train.describe())
display(train.describe(include="object"))

'기술통계 확인'

,age,income,visits
count,20.000000,20.000000,20.000000
mean,36.550000,365.500000,5.400000
std,10.143575,101.435746,2.233713
min,22.000000,220.000000,2.000000
25%,27.750000,277.500000,3.750000
50%,36.000000,360.000000,5.500000
75%,44.250000,442.500000,7.000000
max,55.000000,550.000000,9.000000


C:\Users\user\AppData\Local\Temp\ipykernel_18204\3325191646.py:4: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  display(train.describe(include="object"))


,purchase
count,20
unique,2
top,예
freq,12


In [4]:
# 4. 컬럼명 및 데이터 타입 확인
display('컬럼명 및 데이터 타입 확인')
display(train.columns)
display(train.dtypes)

'컬럼명 및 데이터 타입 확인'

Index(['age', 'income', 'visits', 'purchase'], dtype='str')

age         int64
income      int64
visits      int64
purchase      str
dtype: object

In [5]:
# 5. 결측치 확인
display('결측치 확인')
display(train.isnull().sum())
display(train.isnull().sum().sum())

'결측치 확인'

age         0
income      0
visits      0
purchase    0
dtype: int64

np.int64(0)

In [6]:
# 6. 결측치 비율 확인 (%)
display('결측치 비율 확인')
display((train.isnull().mean() * 100).round(2))

'결측치 비율 확인'

age         0.0
income      0.0
visits      0.0
purchase    0.0
dtype: float64

In [7]:
# 7. 중복 데이터 확인
display('중복 데이터 확인')
display(train.duplicated().sum())

'중복 데이터 확인'

np.int64(0)

In [8]:
# 8. 고유값 개수 확인
display('고유값 개수 확인')
display(train.nunique()) # 범주타입(purchase)만 확인해도 됨

'고유값 개수 확인'

age         20
income      20
visits       8
purchase     2
dtype: int64

In [9]:
# 9. 타깃 변수 분포 확인
# 타깃 컬럼명이 purchase라고 가정
display('타깃 변수 분포 확인')
display(train["purchase"].value_counts())
display(train["purchase"].value_counts(normalize=True).round(3))

'타깃 변수 분포 확인'

purchase
예      12
아니오     8
Name: count, dtype: int64

purchase
예      0.6
아니오    0.4
Name: proportion, dtype: float64

전처리

In [ ]:
# 1. 필요하다면 시각화 탐색

# 2. 독립변수(X)와 종속변수(y) 분리
X = train.drop(columns=["purchase"])
y = train["purchase"]

# 평가용 데이터
X_test = test.copy()

In [11]:
# 3. 종속변수 레이블 인코딩
le = LabelEncoder() # 0, 1 정수로 인코딩
y = le.fit_transform(y)

# 4. 학습 데이터와 검증 데이터 분할
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)